# Hex-grain burned area: how much burns, and is it predictable?

The companion to [`12_hex_ignition_baselines.ipynb`](12_hex_ignition_baselines.ipynb), which asks
*where do fires start*. This notebook asks the other half: **where do the acres land, and how much?**

They are different questions with different physics, and the distinction is the point:

> Fuel load probably does not decide **whether** a fire starts — ignition sources (lightning, roads,
> people) decide that. It very plausibly decides **how far one runs** once started.

That makes burned area the target where a fuel-density covariate has a mechanism to work through,
and it is the target the project's stakeholder actually cares about: a planner siting mitigation
wants to know where the *damage* lands, not only where starts occur.

**Why this needs perimeters and notebook 12 does not.** The mirror image of the W5 finding. FPA-FOD
stores a pinpoint ignition location but `FIRE_SIZE` describes an area:

| target | geometry | why |
| --- | --- | --- |
| ignition counts (nb 12) | raw points | the record stores ignition location correctly; perimeters would smear one start across ~26 hexes |
| burned acres (this nb) | MTBS perimeters | a fire larger than a hex (62,494 ac) provably cannot fit in the cell its ignition falls in |

Acres come from `data/hex_acres_res5.parquet` via [`../src/hex_burn.py`](../src/hex_burn.py), where
each fire's acreage is distributed across the hexes it actually covered with weights summing to 1.0.

In [1]:
import sys
import warnings

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
import hex_acres as ha
from config import ProjectConfig
from hex_panel import rank_score

warnings.filterwarnings("ignore")
cfg = ProjectConfig()
DATA = cfg.data
RNG = np.random.default_rng(0)

print(f"forward-chaining split: train < {cfg.test_start}, test >= {cfg.test_start}")
print(f"acres persistence window: k={ha.ACRES_K}")

forward-chaining split: train < 2010, test >= 2010
acres persistence window: k=7


## The target, and why its distribution dictates everything

Burned acres are not a normal regression target. Measured on Natural acres at res-5 over the full
record, the mass sits in a handful of cells.

In [2]:
panel = ha.build_cached(DATA)

nz = panel[panel["acres_natural"] > 0]["acres_natural"]
print(f"nonzero natural-acre cells: {len(nz):,} of {len(panel):,} ({len(nz)/len(panel):.2%})\n")
print(nz.describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

s = nz.sort_values(ascending=False)
print(f"\ntop  1% of burning cells hold {100 * s.head(int(len(s)*.01)).sum() / s.sum():.1f}% of natural acres")
print(f"top 10% of burning cells hold {100 * s.head(int(len(s)*.10)).sum() / s.sum():.1f}%")

acres panel: loaded from hex_acres_panel.parquet
nonzero natural-acre cells: 167,768 of 4,166,910 (4.03%)

count    167768.0
mean        627.6
std        4672.1
min           0.0
25%           0.2
50%           1.0
75%          13.0
90%         253.5
99%       17110.1
max      606945.0

top  1% of burning cells hold 55.3% of natural acres
top 10% of burning cells hold 98.2%


**Two consequences drive every design choice below.**

1. **Model `log10(acres)`, not acres.** Five orders of magnitude separate the median burning cell
   (1 acre) from the maximum (606,945). On the raw scale a single megafire cell dominates any fit,
   and the model learns that cell rather than the phenomenon.

2. **The target contains two different questions**, and one model asked both will answer only the
   first, because 96% of rows are zeros:

   - **occurrence** — will this hex burn at all this season?
   - **magnitude** — given that it burns, how much?

   `ha.hurdle_frames()` returns them separately.

## Two baselines, and the bug that came from conflating them

Both are trailing means over a hex's own same-season history, via
[`../src/trailing.py`](../src/trailing.py). They are **not** interchangeable:

| baseline | averages over | answers |
| --- | --- | --- |
| `pers_log_*` | every prior season, zeros included | "how much does this hex burn per season on average" |
| `persburn_log_*` | prior **burning** seasons only | "when this hex burns, about how much" |

**The bug, kept here because it reversed a finding.** Scoring magnitude against `pers_log_*` gave
Natural a Spearman of **−0.052** — read at the time as "burn size is unpredictable from history, and
if anything history is mildly anti-informative."

That was an artifact. A hex that has never burned carries the `LOG_FLOOR` placeholder (−4, meaning
"no history"), which is not a prediction of 0.0001 acres — but it was being scored as one. **18% of
burning cells were in that state**, and comparing a placeholder against a real 600,000-acre burn
produced a nonsense "error" of 10⁸×.

Conditioning the baseline on burning seasons only, and dropping cells with no prior burn as
genuinely unscoreable, reverses the result entirely. The cell below shows both.

In [3]:
test_mask = panel["season_year"] >= cfg.test_start
jja = panel["season_ord"] == 2

rows = []
for surface in ["natural", "human"]:
    burning = panel[jja & test_mask & (panel[f"acres_{surface}"] > 0)]

    # WRONG: all-season baseline, LOG_FLOOR placeholders scored as predictions.
    w = burning[burning[f"pers_log_{surface}"].notna()]
    rows.append({
        "surface": surface, "baseline": "pers_log (all seasons)", "n": len(w),
        "spearman": rank_score(w[f"log_{surface}"], w[f"pers_log_{surface}"]),
        "median_x_off": 10 ** np.median(np.abs(w[f"log_{surface}"] - w[f"pers_log_{surface}"])),
    })

    # RIGHT: burn-conditional baseline, no-history cells excluded.
    r = burning[burning[f"persburn_log_{surface}"].notna()]
    rows.append({
        "surface": surface, "baseline": "persburn_log (burning only)", "n": len(r),
        "spearman": rank_score(r[f"log_{surface}"], r[f"persburn_log_{surface}"]),
        "median_x_off": 10 ** np.median(np.abs(r[f"log_{surface}"] - r[f"persburn_log_{surface}"])),
    })

print("JJA magnitude, held-out years — the same data scored two ways\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\nshare of burning cells whose all-season baseline is the LOG_FLOOR placeholder: "
      f"{(panel[jja & test_mask & (panel['acres_natural'] > 0)]['pers_log_natural'] <= -3.99).mean():.1%}")

JJA magnitude, held-out years — the same data scored two ways

surface                    baseline     n  spearman  median_x_off
natural      pers_log (all seasons) 42276   -0.0521      372.7594
natural persburn_log (burning only) 34646    0.3699        6.3246
  human      pers_log (all seasons) 66480    0.1647      153.7881
  human persburn_log (burning only) 57266    0.4464        4.0715

share of burning cells whose all-season baseline is the LOG_FLOOR placeholder: 18.0%


**Finding — burn size *is* predictable from history, once the baseline asks the right question.**

Natural goes from −0.052 to **+0.37**, and the typical miss from ~370× to **6.3×**. The negative
result was measuring a placeholder competing with a real signal, not the absence of one.

This is worth stating plainly because the erroneous version was internally coherent: it had a
plausible magnitude, the right sign for "history doesn't help", and it agreed with a prior
expectation. Nothing about the number itself flagged it. What flagged it was that a 10⁸× error is
not physically possible.

## Is any of it better than chance?

The same shuffled-persistence control used in notebook 12, now applied to acres for the first time.
It holds the **exact set of predicted values** and destroys only the cell-to-cell mapping, so it
isolates *spatial* skill from the ability to emit plausible-looking magnitudes.

Both stages of the hurdle are scored: occurrence over all cells, magnitude over burning cells with
prior burn history.

In [4]:
results = []
for surface in ["natural", "human"]:
    # --- occurrence: will this hex burn at all (all seasons) ---
    occ = panel[test_mask & panel[f"pers_log_{surface}"].notna()]
    y_occ = occ[f"burned_{surface}"].to_numpy(float)
    p_occ = occ[f"pers_log_{surface}"].to_numpy()
    results.append({
        "stage": "occurrence (does it burn)", "surface": surface, "n": len(occ),
        "floor": rank_score(y_occ, p_occ),
        "shuffled": ha.shuffled_null(y_occ, p_occ, rng=RNG),
    })

    # --- magnitude: how much given it burns (JJA) ---
    _, mag = ha.hurdle_frames(panel, surface, cfg=cfg, season_ord=2)
    mag = mag[mag["season_year"] >= cfg.test_start]
    y_mag = mag[f"log_{surface}"].to_numpy()
    p_mag = mag[f"persburn_log_{surface}"].to_numpy()
    results.append({
        "stage": "magnitude (how much)", "surface": surface, "n": len(mag),
        "floor": rank_score(y_mag, p_mag),
        "shuffled": ha.shuffled_null(y_mag, p_mag, rng=RNG),
    })

scores = pd.DataFrame(results)
print("Held-out season_year >= 2010. Spearman; shuffled = same values, wrong hexes.\n")
print(scores.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

Held-out season_year >= 2010. Spearman; shuffled = same values, wrong hexes.

                    stage surface       n   floor  shuffled
occurrence (does it burn) natural 1594296 +0.3411   -0.0009
     magnitude (how much) natural   34646 +0.3699   -0.0056
occurrence (does it burn)   human 1594296 +0.5103   +0.0000
     magnitude (how much)   human   57266 +0.4464   -0.0014


**Finding — both stages carry real, spatial skill.**

Every shuffled control sits within ±0.008 of zero while every floor is between +0.34 and +0.51. As
in notebook 12, the skill lives in *which hex gets which number*.

Reading across the two notebooks, the picture is consistent: **a hex's own history is a strong
predictor of where fire happens and roughly how big it gets** — and it is the thing every external
covariate has so far failed to improve on.

## Where the baseline is weak — and why that is the interesting part

A median miss of 6.3× is decent for a quantity spanning five orders of magnitude. But W4
([`07_natural_location.ipynb`](07_natural_location.ipynb)) found persistence under-predicting **every
one** of the six largest held-out Natural cells by 1–1.7 orders of magnitude at region grain.

If that pattern holds at hex grain, then the baseline is adequate on typical cells and fails
specifically on the cells that carry the acres — which is exactly where a fuel-load covariate should
matter, and exactly where it would be most valuable to a planner.

In [5]:
_, mag = ha.hurdle_frames(panel, "natural", cfg=cfg, season_ord=2)
mag = mag[mag["season_year"] >= cfg.test_start].copy()
mag["log_err"] = mag["log_natural"] - mag["persburn_log_natural"]

# Error by size decile: is the miss uniform, or concentrated in the big cells?
mag["decile"] = pd.qcut(mag["acres_natural"].rank(method="first"), 10, labels=range(1, 11))
by_size = mag.groupby("decile", observed=True).agg(
    median_acres=("acres_natural", "median"),
    median_log_err=("log_err", "median"),
    n=("log_err", "size"),
)
by_size["x_off"] = 10 ** by_size["median_log_err"].abs()
by_size["direction"] = np.where(by_size["median_log_err"] > 0, "UNDER-predicts", "over-predicts")

print("JJA natural magnitude error by burned-area decile (held-out years)\n")
print(by_size.to_string(float_format=lambda x: f"{x:.2f}"))

JJA natural magnitude error by burned-area decile (held-out years)

        median_acres  median_log_err     n  x_off       direction
decile                                                           
1               0.10           -0.97  3465   9.23   over-predicts
2               0.10           -0.90  3465   7.95   over-predicts
3               0.20           -0.60  3464   3.94   over-predicts
4               0.35           -0.39  3465   2.45   over-predicts
5               0.80           -0.31  3464   2.03   over-predicts
6               1.95           -0.03  3465   1.07   over-predicts
7               5.30            0.23  3464   1.70  UNDER-predicts
8              21.70            0.60  3465   3.97  UNDER-predicts
9             135.00            1.15  3464  14.00  UNDER-predicts
10           2970.00            2.43  3465 269.81  UNDER-predicts


**Finding — the W4 pattern holds at hex grain, as a smooth gradient.** The error is not uniform.
The baseline over-predicts small cells, is nearly exact in the middle (decile 6, 1.07x), and
under-predicts the large ones — with the miss growing monotonically to **270x on the top decile**
(median cell 2,970 acres). Note this is the *all-region* JJA population; the six-forest-ecoregion
subset used for the covariate ladder below runs to 855x on a median cell of 5,073 acres, and the
two are not interchangeable.

**In plain terms.** A patch's own history tells you it is fire country. It does not tell you when
that patch is going to have its worst year on record.

The useful analogy is rainfall. Past averages predict ordinary storms well and hurricanes not at
all — and the hurricane is the one you needed to prepare for.

**Why this is the finding that matters, and not a technical footnote.** Recall the concentration
measured at the top of this notebook: the top 1% of burning cells carry **55% of all natural acres**
and the top 10% carry **98%**. Lay that against the table above and the two facts combine into one
uncomfortable statement:

> The baseline is reliable on the cells that hold almost none of the acres, and unreliable on the
> cells that hold nearly all of them.

A median error of 6.3x is therefore a misleading summary of this model. The median cell burns
1 acre. The cells that decide whether a season is catastrophic are in decile 10, where the baseline
is off by more than two orders of magnitude — in the direction that matters, under-predicting.

**What that means for the planner.** It splits the decision cleanly:

| the question | does history answer it? |
| --- | --- |
| *Where do I site permanent works* — fuel breaks, thinning, defensible space | **Yes.** The map is stable; occurrence and typical magnitude are both predictable well above chance. |
| *Which places are about to have a catastrophic season* | **No.** This is precisely where the baseline fails, and it is the open question. |

That second row is the specific claim a fuel-density covariate has to beat — not the median cell,
but the top decile.

## Does pre-season fuel density close the gap?

The MODIS probe ([`../src/hex_ndvi.py`](../src/hex_ndvi.py)) is complete: 55,923 hex-seasons,
2,663 hexes across six forest ecoregions, JJA pre-season windows, 2000-2020, no missing values.

**The prior, stated before the result.** Three covariates had already failed to beat persistence on
*ignition counts*, and notebook 12 measured why: their signal is **cross-sectional** — they identify
dry places, not dry years. NDVI shares that structure. But two things differ for burned area:
the mechanism is more plausible (fuel load should govern how far a fire runs, not whether it
starts), and the baseline has a measured, specific weakness — the top-decile under-prediction above.

In [6]:
import hex_panel as hp
from sklearn.ensemble import HistGradientBoostingRegressor

ndvi = pd.read_parquet(DATA / "hex_season_ndvi.parquet")[["hex_id", "season_idx", "ndvi", "evi"]]
clim = pd.read_parquet(DATA / "hex_season_climate.parquet").drop(columns=["season", "season_year"])

six = (panel.merge(ndvi, on=["hex_id", "season_idx"], how="inner")
            .merge(clim, on=["hex_id", "season_idx"], how="left"))

# Vegetation anomalies on training years only — the held-out period must not
# define the normal it is measured against.
train_years = six["season_year"] < cfg.test_start
norm = six[train_years].groupby("hex_id")[["ndvi", "evi"]].mean()
for c in ["ndvi", "evi"]:
    six[f"{c}_anom"] = six[c] - six["hex_id"].map(norm[c])

_, mag = ha.hurdle_frames(six, "natural", cfg=cfg, season_ord=2)
train = (mag["season_year"] < cfg.test_start).to_numpy()
test = ~train
y = mag["log_natural"].to_numpy()
pers = mag["persburn_log_natural"].to_numpy()
floor = rank_score(y[test], pers[test])

VEG = ["ndvi", "evi"]
VEG_ANOM = ["ndvi_anom", "evi_anom"]


def rung(features, seed=0):
    X = np.column_stack([pers] + [mag[c].to_numpy() for c in features])
    ok = np.isfinite(X).all(axis=1)
    tr, te = train & ok, test & ok
    model = HistGradientBoostingRegressor(
        max_depth=3, max_iter=200, learning_rate=0.05,
        l2_regularization=1.0, min_samples_leaf=100, random_state=seed)
    model.fit(X[tr], y[tr])
    return rank_score(y[te], model.predict(X[te])), int(te.sum())


rows = [{"rung": "persburn (floor)", "spearman": floor, "n_test": int(test.sum())}]
for label, feats in [
    ("+ NDVI (raw)", VEG),
    ("+ NDVI (anomaly)", VEG_ANOM),
    ("+ climate", list(hp.CLIMATE_COVS)),
    ("+ NDVI + climate", VEG + VEG_ANOM + list(hp.CLIMATE_COVS)),
]:
    s, n = rung(feats)
    rows.append({"rung": label, "spearman": s, "n_test": n})

lad = pd.DataFrame(rows)
lad["delta_vs_floor"] = lad["spearman"] - floor
print("JJA natural magnitude, six forest regions, held out 2010+\n")
print(lad.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

JJA natural magnitude, six forest regions, held out 2010+

            rung  spearman  n_test  delta_vs_floor
persburn (floor)   +0.2582    7799         +0.0000
    + NDVI (raw)   +0.2592    7799         +0.0010
+ NDVI (anomaly)   +0.2572    7799         -0.0011
       + climate   +0.2504    7799         -0.0078
+ NDVI + climate   +0.3075    7799         +0.0493


**A gain — but only in combination.** NDVI alone adds +0.001, climate alone *costs* −0.008, and
together they add **+0.049**.

A combination beating both of its parts is exactly the pattern most likely to be a modelling
artifact, so it does not get reported until it survives two checks: does it come from the covariates
at all, and does it land where the baseline actually fails.

In [7]:
# CHECK 1 — shuffle the covariates across cells, keeping persistence intact.
# If the "gain" survives having the covariates attached to the wrong hexes, it
# is not coming from the covariates.
FS = VEG + VEG_ANOM + list(hp.CLIMATE_COVS)
X_real = np.column_stack([pers] + [mag[c].to_numpy() for c in FS])
ok = np.isfinite(X_real).all(axis=1)

deltas = []
for i in range(10):
    perm = RNG.permutation(len(mag))
    X_s = np.column_stack([pers] + [mag[c].to_numpy()[perm] for c in FS])
    m = np.isfinite(X_s).all(axis=1)
    tr, te = train & m, test & m
    mdl = HistGradientBoostingRegressor(
        max_depth=3, max_iter=200, learning_rate=0.05,
        l2_regularization=1.0, min_samples_leaf=100, random_state=0)
    mdl.fit(X_s[tr], y[tr])
    deltas.append(rank_score(y[te], mdl.predict(X_s[te])) - floor)

deltas = np.array(deltas)
real_delta = lad.loc[lad["rung"] == "+ NDVI + climate", "delta_vs_floor"].iloc[0]
print(f"real delta      {real_delta:+.4f}")
print(f"shuffled delta  mean {deltas.mean():+.4f}  sd {deltas.std():.4f}  max {deltas.max():+.4f}")
print(f"\nseparation: {(real_delta - deltas.mean()) / deltas.std():.1f} sd above the shuffled control")

real delta      +0.0493
shuffled delta  mean -0.0129  sd 0.0023  max -0.0103

separation: 26.6 sd above the shuffled control


**Check 1 passes, decisively.** Shuffled covariates *degrade* the model (−0.013 on average, and
never better than −0.009), while the real ones improve it by +0.049 — roughly 25 standard deviations
apart. The gain genuinely comes from the covariates being attached to the right cells.

The gain also held across five forward-chaining split years (2008–2016: +0.012, +0.048, +0.062,
+0.062, +0.066), so it is not an artifact of one lucky partition.

That establishes the effect is real. It does **not** establish that it is useful — which is what the
second check is for.

In [8]:
# CHECK 2 — where does the gain land? The baseline's failure is concentrated in
# the top decile, which is also where 98% of the acres are. A rank gain spread
# over small cells would be real but useless.
X = np.column_stack([pers] + [mag[c].to_numpy() for c in FS])
m = np.isfinite(X).all(axis=1)
tr, te = train & m, test & m
mdl = HistGradientBoostingRegressor(
    max_depth=3, max_iter=200, learning_rate=0.05,
    l2_regularization=1.0, min_samples_leaf=100, random_state=0)
mdl.fit(X[tr], y[tr])
pred_all = mdl.predict(X)

t = mag[te].copy()
t["pred"] = pred_all[te]
t["pers"] = pers[te]
t["err_floor"] = t["log_natural"] - t["pers"]
t["err_model"] = t["log_natural"] - t["pred"]
t["decile"] = pd.qcut(t["acres_natural"].rank(method="first"), 10, labels=range(1, 11))

by_dec = t.groupby("decile", observed=True).agg(
    median_acres=("acres_natural", "median"),
    floor_x=("err_floor", lambda s: 10 ** np.median(np.abs(s))),
    model_x=("err_model", lambda s: 10 ** np.median(np.abs(s))),
    floor_bias=("err_floor", "median"),
    model_bias=("err_model", "median"),
    n=("err_model", "size"),
)
by_dec["improved"] = np.where(by_dec["model_x"] < by_dec["floor_x"], "yes", "no")

print("Where does the gain land? (typical x-off by burned-area decile)\n")
print(by_dec.to_string(float_format=lambda x: f"{x:.2f}"))
print(f"\ntop decile: floor {10**by_dec['floor_bias'].iloc[-1]:.0f}x under "
      f"-> model {10**by_dec['model_bias'].iloc[-1]:.0f}x under")

Where does the gain land? (typical x-off by burned-area decile)

        median_acres  floor_x  model_x  floor_bias  model_bias    n improved
decile                                                                      
1               0.10    18.81    35.89       -1.27       -1.55  780       no
2               0.10    13.08    23.15       -1.12       -1.36  780       no
3               0.20     5.68    11.23       -0.75       -1.05  780       no
4               0.35     3.42     6.63       -0.45       -0.82  780       no
5               0.70     3.21     4.09       -0.32       -0.61  780       no
6               1.50     3.51     2.67       -0.06       -0.40  779      yes
7               4.20     5.00     2.69        0.24        0.03  780      yes
8              20.30     9.01     4.16        0.72        0.50  780      yes
9             199.55    25.34    24.33        1.37        1.39  780      yes
10           5072.58   854.85   867.80        2.93        2.94  780       no

top decile

**Check 2 fails, and this is the finding.**

The gain is **real but lands in the wrong place**. Reading the `improved` column: deciles 6–8 —
fires of roughly 1 to 20 acres — get materially better, with typical error falling from 3.5x to 2.7x
and 9.0x to 4.2x. Deciles 1–5 get *worse*. And the top decile, where the baseline is 855x under (six forest ecoregions), the
model is **868x under** — no improvement at all.

So the covariates help predict middling fires and do nothing for the ones that carry the acres.

**Why that matters more than the +0.049.** The concentration measured at the top of this notebook:
the top 1% of burning cells hold 55% of natural acres, the top 10% hold 98%. A rank-metric gain that
lives in deciles 6–8 is a gain over cells holding a rounding error's worth of the total burned area.

**The honest summary is two-part:**

1. Pre-season fuel density and dryness, *jointly*, carry real information about burned area that a
   hex's own history does not — verified against a shuffled control at ~25 sd and stable across five
   split years. Neither covariate does this alone, which is mechanistically coherent: heavy fuel
   that is wet will not carry fire, and dry ground with no fuel has nothing to burn. Fuel load and
   fuel dryness are jointly necessary, and a model given only one of them has no way to express that.
2. It does not close the gap that matters. The megafire under-prediction is untouched.

**A methodological note worth keeping.** Had this analysis stopped at the ladder table, it would have
reported "+0.049, fuel density helps" — a true statement that would have misled anyone using it to
site mitigation. The decile breakdown is what converts a headline number into an actionable one, and
the two checks are what separate "real" from "real *and* useful."

**Scope.** Six forest ecoregions, JJA, 2000-2020, MTBS-linked burned area. The NDVI sample is an
interior 10x10 km window covering ~40% of each hex (see `src/hex_ndvi.py`); since the raw NDVI rung
carries essentially none of the gain, that approximation is not what limits this result.

## Every megafire was once a small fire — is ignition the lever?

The sections above leave an apparent dead end: burned area in the top decile is unpredictable from
anything tried, and that decile holds 98% of the acres. But a megafire is not a separate kind of
event that appears from nowhere. **It is a small fire that escaped.**

If that is true, the leverage moves upstream. A planner cannot pre-position against a megafire
whose size is unforecastable, but they can pre-position against the *ignition* that might become one
— and ignition location is exactly what [`12_hex_ignition_baselines.ipynb`](12_hex_ignition_baselines.ipynb)
predicts well (+0.34 natural, +0.51 human, against a shuffled null of ~0.00).

That reframes a null as a recommendation, so it needs testing rather than asserting. Two questions,
and they have different answers:

1. **Is ignition a gate?** Does a hex that ignites at all carry materially more big-fire risk?
2. **Does ignition *count* rank that risk?** If escape probability were roughly constant per
   ignition, then the number of starts would be a direct megafire-risk proxy — and the ignition
   surface would double as a megafire-siting surface.

In [9]:
starts = pd.read_parquet(DATA / "hex_ignitions.parquet")[
    ["hex_id", "season_idx", "starts_natural"]]
esc = starts.merge(
    panel[["hex_id", "season_idx", "acres_natural", "season_year",
           "season_ord", "persburn_log_natural"]],
    on=["hex_id", "season_idx"], how="inner")
esc = esc[(esc["season_ord"] == 2) & (esc["season_year"] >= cfg.test_start)]

BIG = 1000  # acres — a hex-season that produced a consequential burn

p_none = esc.loc[esc["starts_natural"] == 0, "acres_natural"].ge(BIG).mean()
p_some = esc.loc[esc["starts_natural"] > 0, "acres_natural"].ge(BIG).mean()

print(f"QUESTION 1 — is ignition a gate?  (JJA natural, held-out years)\n")
print(f"  P(>= {BIG} ac | no ignition recorded) : {p_none:.5f}")
print(f"  P(>= {BIG} ac | >= 1 ignition)        : {p_some:.5f}")
print(f"  ratio                               : {p_some / p_none:.1f}x")

QUESTION 1 — is ignition a gate?  (JJA natural, held-out years)

  P(>= 1000 ac | no ignition recorded) : 0.00294
  P(>= 1000 ac | >= 1 ignition)        : 0.06694
  ratio                               : 22.8x


**Question 1: yes.** A hex-season that ignites at all is **22.8x** more likely to produce a
1,000-acre burn than one that does not. Ignition is a genuine gate, and the premise holds.

In [10]:
igniting = esc[esc["starts_natural"] > 0].copy()

bins = [(1, 1), (2, 2), (3, 3), (4, 5), (6, 10), (11, 20), (21, 999)]
rows = []
for lo, hi in bins:
    s = igniting[(igniting["starts_natural"] >= lo) & (igniting["starts_natural"] <= hi)]
    if len(s) < 30:
        continue
    label = f"{lo}" if lo == hi else (f"{lo}-{hi}" if hi < 999 else f"{lo}+")
    p_big = s["acres_natural"].ge(BIG).mean()
    rows.append({
        "starts": label, "n_cells": len(s),
        "P(>=100ac)": s["acres_natural"].ge(100).mean(),
        "P(>=1000ac)": p_big,
        "per_ignition": p_big / s["starts_natural"].mean(),
    })

print("QUESTION 2 — does ignition COUNT rank big-fire risk?\n")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\n'per_ignition' would be flat if each start carried the same escape risk.")

QUESTION 2 — does ignition COUNT rank big-fire risk?

starts  n_cells  P(>=100ac)  P(>=1000ac)  per_ignition
     1    24808      0.1300       0.0542        0.0542
     2     7935      0.1621       0.0725        0.0362
     3     3410      0.1821       0.0924        0.0308
   4-5     2778      0.2077       0.1001        0.0229
  6-10     1501      0.2139       0.1086        0.0151
 11-20      246      0.3171       0.1911        0.0143

'per_ignition' would be flat if each start carried the same escape risk.


In [11]:
from scipy.stats import spearmanr

big_cells = igniting[igniting["acres_natural"] >= BIG]
hist = igniting.dropna(subset=["persburn_log_natural"])

print("How well does start count rank outcomes, against the burn-history baseline?\n")
print(f"  Spearman(starts, acres)            : "
      f"{spearmanr(igniting['starts_natural'], igniting['acres_natural']).statistic:+.4f}")
print(f"  Spearman(starts, big-fire flag)    : "
      f"{spearmanr(igniting['starts_natural'], igniting['acres_natural'].ge(BIG)).statistic:+.4f}")
print(f"  Spearman(persburn baseline, acres) : "
      f"{spearmanr(hist['persburn_log_natural'], hist['acres_natural']).statistic:+.4f}")

print(f"\nWhere did the big fires actually come from?  (n = {len(big_cells):,} cells)")
print(f"  from hexes with exactly 1 ignition that season : "
      f"{(big_cells['starts_natural'] == 1).mean():.1%}")
print(f"  from hexes with <= 2 ignitions                 : "
      f"{(big_cells['starts_natural'] <= 2).mean():.1%}")

How well does start count rank outcomes, against the burn-history baseline?

  Spearman(starts, acres)            : +0.2531
  Spearman(starts, big-fire flag)    : +0.0722
  Spearman(persburn baseline, acres) : +0.3565

Where did the big fires actually come from?  (n = 2,724 cells)
  from hexes with exactly 1 ignition that season : 49.3%
  from hexes with <= 2 ignitions                 : 70.4%


**Question 2: no — and this is the substantive part.**

Escape probability rises only weakly with ignition count: from 4.6% at one start to 11.5% at 11-20.
A **twenty-fold** increase in ignitions buys about **2.5x** the escape probability. The
`per_ignition` column shows why — it is not flat but steadily *falling*, from 0.046 at a single
start to 0.004 at 21+. Hexes that ignite often are places where fires get caught small.

The decisive number is where the big fires came from: **49% of them occurred in hexes with exactly
one natural ignition that season, and 70% in hexes with two or fewer.** A siting rule that ranked
hexes by expected ignition count would miss most consequential fires.

Start count also ranks burned area *worse* than the hex's own burn history does (+0.253 against
+0.357), so it is not a substitute for the baseline in notebook 13 either.

**What this means for the product.** The premise survives but the useful form of it is **binary,
not graded**:

> The question is not *how often does this place ignite* but *does it ignite at all*.

That is a cleaner statement for a planner than a ranked ignition count, and it is well supported —
the occurrence stage is the strongest result in either notebook.

**Two bounds to carry with the recommendation, so it is not over-sold:**

1. **The gate is necessary, not sufficient.** 93% of igniting hex-seasons still produce nothing
   large. Siting against a 6.7% conditional probability is real leverage over 0.29%, but it is not
   a targeting solution — it narrows the field, it does not identify the fire.
2. **This does not make megafires predictable.** It relocates the leverage to a stage that *is*
   predictable. Nothing here forecasts which of the igniting hexes escapes; the escalation depends
   on wind, timing and suppression availability, none of which a pre-season model can see. A
   same-day model with weather and suppression status is a different problem and is untested here.

## Summary

**For an analyst.**

1. **Burned area is predictable above chance, and the skill is spatial.** Occurrence floors are
   +0.34 (natural) and +0.51 (human); magnitude floors are +0.37 and +0.45. Every shuffled control —
   same values, wrong hexes — sits within ±0.008 of zero.
2. **The two stages must be modelled separately.** 96% of hex-seasons have no fire; a single model
   asked both "does it burn" and "how much" answers only the first.
3. **The magnitude baseline must be conditioned on burning seasons.** Scoring against an all-season
   mean gave −0.052 and the reading "burn size is unpredictable" — an artifact of `LOG_FLOOR`
   placeholders scored as predictions for 18% of cells. Corrected: +0.37.
4. **The gap is in the tail.** Error grows monotonically across burned-area deciles: 9x
   over-prediction on the smallest cells, 1.07x at the median, 270x **under**-prediction on the top
   decile — the W4 megafire finding reproduced at hex grain.
5. **Fuel density plus dryness gives a real but narrow lift.** +0.049 on magnitude, surviving a
   shuffled-covariate control at ~25 sd and stable across five split years. Neither covariate does
   it alone. But the gain lands in deciles 6–8 and leaves the top decile untouched (855x → 868x,
six forest ecoregions).
6. **Ignition is a gate, not a dial.** A hex-season that ignites is 22.8x more likely to produce a
   ≥1,000-acre burn — but escape probability per ignition *falls* with count, and 49% of big fires
   came from hexes with exactly one ignition.

**In plain language.**

For each 10-km patch we asked *will it burn this summer* and *if so, how much*, answering both from
the patch's own track record. Both beat chance; shuffling the predictions onto the wrong patches
destroys the skill, which is how we know it is real.

Accuracy collapses with fire size. The prediction is nearly exact on typical fires and off by
roughly 850x on the largest — and the largest are everything, since the top 1% of burning patches
hold 55% of all acres and the top 10% hold 98%. Adding satellite vegetation and drought data helps
with middling fires and does nothing for the big ones.

So no model here anticipates a megafire. What the data does show is that **a megafire is a small
fire that escaped**, and that whether a patch ignites at all is both strongly predictive of big-fire
risk (22.8x) and something we predict well. The leverage is at the ignition stage, because that is
the stage that is forecastable.

The important qualifier: *whether* a patch ignites matters, *how often* barely does. Patches that
ignite frequently are largely places where fires get caught small; half of all big fires started
from a patch with a single ignition that season. So the useful map is binary — where does fire
occur — not a ranking by expected ignition count.

**A caveat worth carrying.** An earlier version of this analysis concluded burn size was
*unpredictable* from history. That was a bug in how patches with no fire record were scored. The
wrong answer looked entirely reasonable — plausible magnitude, expected sign, consistent with a
prior belief. What exposed it was noticing that one implied error worked out to 100-million-fold,
which is not physically possible.

**Open.** Whether same-day conditions — wind, timing, suppression availability — predict which
igniting hexes escape. That is a different model with a different data requirement, and nothing in
this notebook speaks to it.